In [ ]:
from pathlib import Path

def carpeta_proyecto(nombre, marca):
    """Localiza los CSV en Jupyter, Visual Studio Code y Colab."""
    from pathlib import Path
    import json

    saltar = {
        ".config", "sample_data", "drive", ".git", "__pycache__",
        "node_modules", ".ipynb_checkpoints", "site-packages",
    }

    def directorio_notebook():
        try:
            return Path(__vsc_ipynb_file__).resolve().parent
        except NameError:
            pass
        try:
            from urllib.request import urlopen
            try:
                from ipykernel.connect import get_connection_file
            except ImportError:
                from ipykernel import get_connection_file
            conexion = Path(get_connection_file())
            kernel_id = conexion.name.replace("kernel-", "").replace(".json", "")
            servidores = list(conexion.parent.glob("jpserver-*.json"))[:3]
            for servidor in servidores:
                try:
                    info = json.loads(servidor.read_text(encoding="utf-8"))
                except (OSError, json.JSONDecodeError):
                    continue
                url = str(info.get("url", "")).rstrip("/")
                raiz = info.get("root_dir")
                if not url or not raiz:
                    continue
                api = url + "/api/sessions"
                token = info.get("token") or ""
                if token:
                    api += "?token=" + token
                try:
                    with urlopen(api, timeout=0.8) as respuesta:
                        sesiones = json.loads(respuesta.read().decode())
                except Exception:
                    continue
                for sesion in sesiones:
                    kid = (sesion.get("kernel") or {}).get("id")
                    if kid != kernel_id:
                        continue
                    cuaderno = sesion.get("notebook") or {}
                    relativo = cuaderno.get("path") or sesion.get("path")
                    if relativo:
                        return (Path(raiz) / relativo).resolve().parent
        except Exception:
            pass
        return None

    bases = []
    notebook = directorio_notebook()
    if notebook is not None:
        bases.append(notebook)
    bases.append(Path.cwd().resolve())
    content = Path("/content")
    if content.is_dir():
        bases.append(content)
        drive = content / "drive" / "MyDrive"
        if drive.is_dir():
            bases.append(drive)

    nombradas = []
    sueltas = []
    vistos = set()

    def considerar(directorio, aceptar_suelto):
        if directorio in vistos:
            return
        vistos.add(directorio)
        try:
            carpeta = directorio / nombre
            if (carpeta / marca).is_file():
                nombradas.append(carpeta)
                return
            if directorio.name == nombre and (directorio / marca).is_file():
                nombradas.append(directorio)
                return
            if aceptar_suelto and (directorio / marca).is_file():
                sueltas.append(directorio)
        except OSError:
            return

    def hijos(directorio, profundidad, limite):
        if profundidad < 0 or limite[0] <= 0 or not directorio.is_dir():
            return
        try:
            entradas = list(directorio.iterdir())
        except OSError:
            return
        for hijo in entradas:
            if not hijo.is_dir() or hijo.name.startswith(".") or hijo.name in saltar:
                continue
            limite[0] -= 1
            considerar(hijo, False)
            if nombradas or limite[0] <= 0:
                return
            hijos(hijo, profundidad - 1, limite)
            if nombradas or limite[0] <= 0:
                return

    for base in bases:
        considerar(base, True)
        if nombradas:
            break
        for padre in list(base.parents)[:6]:
            considerar(padre, False)
            if nombradas:
                break
        if nombradas:
            break
        profundidad = 3 if base.name in {"content", "MyDrive"} else 2
        hijos(base, profundidad, [400])
        if nombradas:
            break

    if nombradas:
        return nombradas[0]
    if sueltas:
        return sueltas[0]
    raise FileNotFoundError(
        "No se encuentra " + marca + ". "
        "Tiene que existir la carpeta '" + nombre + "' en el proyecto, "
        "o el archivo en el directorio de trabajo. En Colab puede estar junto a sample_data."
    )




ORIGINALES = carpeta_proyecto("datos originales", "cancion.csv")
print("Originales:", ORIGINALES)
try:
    PREPROCESADOS = carpeta_proyecto("datos preprocesados", "df_macro.csv")
except FileNotFoundError:
    if ORIGINALES.name == "datos originales":
        PREPROCESADOS = ORIGINALES.parent / "datos preprocesados"
    else:
        PREPROCESADOS = ORIGINALES / "datos preprocesados"
    PREPROCESADOS.mkdir(parents=True, exist_ok=True)
print("Preprocesados:", PREPROCESADOS)

import pandas as pd
import numpy as np

df_cancion = pd.read_csv(ORIGINALES / "cancion.csv")
df_features_30 = pd.read_csv(ORIGINALES / "features_30_sec.csv")
df_features_3 = pd.read_csv(ORIGINALES / "features_3_sec.csv")

In [ ]:
print(df_cancion.columns)
print(df_features_30.columns)

Index(['id_cancion', 'cancion', 'duracion', 'coste', 'id_genero', 'id_disco',
       'id_autor', 'id_discografica'],
      dtype='object')
Index(['filename', 'length', 'chroma_stft_mean', 'chroma_stft_var', 'rms_mean',
       'rms_var', 'spectral_centroid_mean', 'spectral_centroid_var',
       'spectral_bandwidth_mean', 'spectral_bandwidth_var', 'rolloff_mean',
       'rolloff_var', 'zero_crossing_rate_mean', 'zero_crossing_rate_var',
       'harmony_mean', 'harmony_var', 'perceptr_mean', 'perceptr_var', 'tempo',
       'mfcc1_mean', 'mfcc1_var', 'mfcc2_mean', 'mfcc2_var', 'mfcc3_mean',
       'mfcc3_var', 'mfcc4_mean', 'mfcc4_var', 'mfcc5_mean', 'mfcc5_var',
       'mfcc6_mean', 'mfcc6_var', 'mfcc7_mean', 'mfcc7_var', 'mfcc8_mean',
       'mfcc8_var', 'mfcc9_mean', 'mfcc9_var', 'mfcc10_mean', 'mfcc10_var',
       'mfcc11_mean', 'mfcc11_var', 'mfcc12_mean', 'mfcc12_var', 'mfcc13_mean',
       'mfcc13_var', 'mfcc14_mean', 'mfcc14_var', 'mfcc15_mean', 'mfcc15_var',
       'mfcc16_mean', 

Se muestran los nombres de las columnas de los DataFrames para entender su contenido.

In [ ]:
display(df_cancion[['id_cancion', 'cancion', 'id_genero']].head())
display(df_features_30[['filename', 'label']].head())

,id_cancion,cancion,id_genero
0,1,blues.00000.wav,1
1,2,blues.00001.wav,1
2,3,blues.00002.wav,1
3,4,blues.00003.wav,1
4,5,blues.00004.wav,1


,filename,label
0,blues.00000.wav,blues
1,blues.00001.wav,blues
2,blues.00002.wav,blues
3,blues.00003.wav,blues
4,blues.00004.wav,blues


In [ ]:
print("Canciones en df_cancion:", df_cancion['cancion'].nunique())
print("Archivos en features_30:", df_features_30['filename'].nunique())

coincidencias = df_cancion['cancion'].isin(df_features_30['filename']).sum()

print("Canciones de df_cancion encontradas en features_30:", coincidencias)

Canciones en df_cancion: 1000
Archivos en features_30: 1000
Canciones de df_cancion encontradas en features_30: 1000


Se renombran las columnas del DataFrame `df_features_30` para evitar conflictos al unirlo con otros datos.

In [ ]:
# Cambiar los nombres de todas las columnas de features_30

df_features_30_ml = df_features_30.copy()

df_features_30_ml = df_features_30_ml.rename(columns={
    col: f"{col}_30s"
    for col in df_features_30_ml.columns
})

Aquí se unen los DataFrames `df_cancion` y `df_features_30_ml` para crear un modelo de datos inicial.

In [ ]:
df_modelo_30 = df_cancion.merge(
    df_features_30_ml,
    left_on='cancion',
    right_on='filename_30s',
    how='inner'
)

In [ ]:
# Guarda la tabla en un archivo CSV

df_modelo_30.to_csv(PREPROCESADOS / "df_modelo_30.csv", index=False)

In [ ]:
#VISTA PREVIA


print("Filas en df_cancion:", df_cancion.shape[0])
print("Filas en df_features_30:", df_features_30.shape[0])
print("Filas tras la unión:", df_modelo_30.shape[0])

print("IDs únicos tras la unión:", df_modelo_30['id_cancion'].nunique())

display(df_modelo_30.head())

Filas en df_cancion: 1000
Filas en df_features_30: 1000
Filas tras la unión: 1000
IDs únicos tras la unión: 1000


,id_cancion,cancion,duracion,coste,id_genero,id_disco,id_autor,id_discografica,filename_30s,length_30s,...,mfcc16_var_30s,mfcc17_mean_30s,mfcc17_var_30s,mfcc18_mean_30s,mfcc18_var_30s,mfcc19_mean_30s,mfcc19_var_30s,mfcc20_mean_30s,mfcc20_var_30s,label_30s
0,1,blues.00000.wav,138.0,744.01,1,1,1,1,blues.00000.wav,661794,...,52.420910,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues
1,2,blues.00001.wav,142.0,4178.85,1,2,2,2,blues.00001.wav,661794,...,55.356403,-0.731125,60.314529,0.295073,48.120598,-0.283518,51.106190,0.531217,45.786282,blues
2,3,blues.00002.wav,96.0,3226.40,1,3,4,3,blues.00002.wav,661794,...,40.598766,-7.729093,47.639427,-1.816407,52.382141,-3.439720,46.639660,-2.231258,30.573025,blues
3,4,blues.00003.wav,111.0,3618.77,1,4,6,4,blues.00003.wav,661794,...,44.427753,-3.319597,50.206673,0.636965,37.319130,-0.619121,37.259739,-3.407448,31.949339,blues
4,5,blues.00004.wav,191.0,1990.81,1,5,7,5,blues.00004.wav,661794,...,86.099236,-5.454034,75.269707,-0.916874,53.613918,-4.404827,62.910812,-11.703234,55.195160,blues


Se verifica que ninguna canción se haya perdido durante el proceso de unión de los DataFrames.

In [ ]:
ids_perdidos = set(df_cancion['id_cancion']) - set(df_modelo_30['id_cancion'])

print("IDs perdidos en la unión:", len(ids_perdidos))


IDs perdidos en la unión: 0


Esta celda muestra las primeras filas del DataFrame `df_features_3`.

In [ ]:
display(df_features_3)

,filename,length,chroma_stft_mean,chroma_stft_var,rms_mean,rms_var,spectral_centroid_mean,spectral_centroid_var,spectral_bandwidth_mean,spectral_bandwidth_var,...,mfcc16_var,mfcc17_mean,mfcc17_var,mfcc18_mean,mfcc18_var,mfcc19_mean,mfcc19_var,mfcc20_mean,mfcc20_var,label
0,blues.00000.0.wav,66149,0.335406,0.091048,0.130405,0.003521,1773.065032,167541.630869,1972.744388,117335.771563,...,39.687145,-3.241280,36.488243,0.722209,38.099152,-5.050335,33.618073,-0.243027,43.771767,blues
1,blues.00000.1.wav,66149,0.343065,0.086147,0.112699,0.001450,1816.693777,90525.690866,2010.051501,65671.875673,...,64.748276,-6.055294,40.677654,0.159015,51.264091,-2.837699,97.030830,5.784063,59.943081,blues
2,blues.00000.2.wav,66149,0.346815,0.092243,0.132003,0.004620,1788.539719,111407.437613,2084.565132,75124.921716,...,67.336563,-1.768610,28.348579,2.378768,45.717648,-1.938424,53.050835,2.517375,33.105122,blues
3,blues.00000.3.wav,66149,0.363639,0.086856,0.132565,0.002448,1655.289045,111952.284517,1960.039988,82913.639269,...,47.739452,-3.841155,28.337118,1.218588,34.770935,-3.580352,50.836224,3.630866,32.023678,blues
4,blues.00000.4.wav,66149,0.335579,0.088129,0.143289,0.001701,1630.656199,79667.267654,1948.503884,60204.020268,...,30.336359,0.664582,45.880913,1.689446,51.363583,-3.392489,26.738789,0.536961,29.146694,blues
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9985,rock.00099.5.wav,66149,0.349126,0.080515,0.050019,0.000097,1499.083005,164266.886443,1718.707215,85931.574523,...,42.485981,-9.094270,38.326839,-4.246976,31.049839,-5.625813,48.804092,1.818823,38.966969,rock
9986,rock.00099.6.wav,66149,0.372564,0.082626,0.057897,0.000088,1847.965128,281054.935973,1906.468492,99727.037054,...,32.415203,-12.375726,66.418587,-3.081278,54.414265,-11.960546,63.452255,0.428857,18.697033,rock
9987,rock.00099.7.wav,66149,0.347481,0.089019,0.052403,0.000701,1346.157659,662956.246325,1561.859087,138762.841945,...,78.228149,-2.524483,21.778994,4.809936,25.980829,1.775686,48.582378,-0.299545,41.586990,rock
9988,rock.00099.8.wav,66149,0.387527,0.084815,0.066430,0.000320,2084.515327,203891.039161,2018.366254,22860.992562,...,28.323744,-5.363541,17.209942,6.462601,21.442928,2.354765,24.843613,0.675824,12.787750,rock


In [ ]:
# El dataset features 3 tiene cada cancion troceada en fragmentos, ademas los nombres de la cancion no coinciden con los otros data set, creamos la columna cancion_base
# para identificar a que cancion pertenece cada trozo y poder hacer el merge. Solo se añade una columna no se borra nada.

df_features_3_ml = df_features_3.copy()
df_features_3_ml['cancion_base'] = (
    df_features_3_ml['filename']
    .str.rsplit('.', n=2)
    .str[0] + '.wav'
)

display(df_features_3_ml[['filename', 'cancion_base', 'label']].head())

,filename,cancion_base,label
0,blues.00000.0.wav,blues.00000.wav,blues
1,blues.00000.1.wav,blues.00000.wav,blues
2,blues.00000.2.wav,blues.00000.wav,blues
3,blues.00000.3.wav,blues.00000.wav,blues
4,blues.00000.4.wav,blues.00000.wav,blues


In [ ]:
# Cuántos trozos tiene cada canción

trozos_por_cancion = (
    df_features_3_ml
    .groupby('cancion_base')
    .size()
    .reset_index(name='num_trozos')
)

display(trozos_por_cancion.head(20))

,cancion_base,num_trozos
0,blues.00000.wav,10
1,blues.00001.wav,10
2,blues.00002.wav,10
3,blues.00003.wav,10
4,blues.00004.wav,10
5,blues.00005.wav,10
6,blues.00006.wav,10
7,blues.00007.wav,10
8,blues.00008.wav,10
9,blues.00009.wav,10


In [ ]:
# Resumen cuántos trozos suelen tener las canciones

print("Número de canciones:", trozos_por_cancion['cancion_base'].nunique())
print("Mínimo de trozos por canción:", trozos_por_cancion['num_trozos'].min())
print("Máximo de trozos por canción:", trozos_por_cancion['num_trozos'].max())
print("Media de trozos por canción:", trozos_por_cancion['num_trozos'].mean())

display(trozos_por_cancion['num_trozos'].value_counts().sort_index())

Número de canciones: 1000
Mínimo de trozos por canción: 9
Máximo de trozos por canción: 10
Media de trozos por canción: 9.99


,count
num_trozos,
9,10
10,990


Se comprueba que cada fragmento de canción se asocie correctamente con su canción completa.

In [ ]:
# Comprobar que cada trozo encuentra su canción

print("Filas en features_3:", df_features_3_ml.shape[0])
print("Canciones distintas en features_3:", df_features_3_ml['cancion_base'].nunique())
print("Canciones en df_modelo_30:", df_modelo_30['cancion'].nunique())

trozos_encontrados = df_features_3_ml['cancion_base'].isin(df_modelo_30['cancion']).sum()

print("Trozos encontrados en df_modelo_30:", trozos_encontrados)

Filas en features_3: 9990
Canciones distintas en features_3: 1000
Canciones en df_modelo_30: 1000
Trozos encontrados en df_modelo_30: 9990


In [ ]:
# Cambiar nombres de las columnas de features_3 para distinguirlos de features_30 cuando hagamos el merge y creemos la macro tabla, si no podrian coincidir los nombres de algunas.

df_features_3_ml = df_features_3_ml.rename(columns={
    col: f"{col}_3s"
    for col in df_features_3_ml.columns
    if col not in ['filename', 'cancion_base', 'label']
})

df_features_3_ml = df_features_3_ml.rename(columns={
    'filename': 'filename_3s',
    'label': 'label_3s'
})

display(df_features_3_ml.head())

,filename_3s,length_3s,chroma_stft_mean_3s,chroma_stft_var_3s,rms_mean_3s,rms_var_3s,spectral_centroid_mean_3s,spectral_centroid_var_3s,spectral_bandwidth_mean_3s,spectral_bandwidth_var_3s,...,mfcc17_mean_3s,mfcc17_var_3s,mfcc18_mean_3s,mfcc18_var_3s,mfcc19_mean_3s,mfcc19_var_3s,mfcc20_mean_3s,mfcc20_var_3s,label_3s,cancion_base
0,blues.00000.0.wav,66149,0.335406,0.091048,0.130405,0.003521,1773.065032,167541.630869,1972.744388,117335.771563,...,-3.241280,36.488243,0.722209,38.099152,-5.050335,33.618073,-0.243027,43.771767,blues,blues.00000.wav
1,blues.00000.1.wav,66149,0.343065,0.086147,0.112699,0.001450,1816.693777,90525.690866,2010.051501,65671.875673,...,-6.055294,40.677654,0.159015,51.264091,-2.837699,97.030830,5.784063,59.943081,blues,blues.00000.wav
2,blues.00000.2.wav,66149,0.346815,0.092243,0.132003,0.004620,1788.539719,111407.437613,2084.565132,75124.921716,...,-1.768610,28.348579,2.378768,45.717648,-1.938424,53.050835,2.517375,33.105122,blues,blues.00000.wav
3,blues.00000.3.wav,66149,0.363639,0.086856,0.132565,0.002448,1655.289045,111952.284517,1960.039988,82913.639269,...,-3.841155,28.337118,1.218588,34.770935,-3.580352,50.836224,3.630866,32.023678,blues,blues.00000.wav
4,blues.00000.4.wav,66149,0.335579,0.088129,0.143289,0.001701,1630.656199,79667.267654,1948.503884,60204.020268,...,0.664582,45.880913,1.689446,51.363583,-3.392489,26.738789,0.536961,29.146694,blues,blues.00000.wav


In [ ]:
# Merge de los 3 notebooks, ahora si podemos porque hemos creado en features_3 la columna cancion_base
# que es el nombre real de la cancion y coincide con el nombre en los otros DF.

df_macro = df_features_3_ml.merge(
    df_modelo_30,
    left_on='cancion_base',
    right_on='cancion',
    how='inner'
)

columnas = ['id_cancion'] + [col for col in df_macro.columns if col != 'id_cancion']
df_macro = df_macro[columnas]

display(df_macro.head())
df_macro.info()

,id_cancion,filename_3s,length_3s,chroma_stft_mean_3s,chroma_stft_var_3s,rms_mean_3s,rms_var_3s,spectral_centroid_mean_3s,spectral_centroid_var_3s,spectral_bandwidth_mean_3s,...,mfcc16_var_30s,mfcc17_mean_30s,mfcc17_var_30s,mfcc18_mean_30s,mfcc18_var_30s,mfcc19_mean_30s,mfcc19_var_30s,mfcc20_mean_30s,mfcc20_var_30s,label_30s
0,1,blues.00000.0.wav,66149,0.335406,0.091048,0.130405,0.003521,1773.065032,167541.630869,1972.744388,...,52.42091,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues
1,1,blues.00000.1.wav,66149,0.343065,0.086147,0.112699,0.001450,1816.693777,90525.690866,2010.051501,...,52.42091,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues
2,1,blues.00000.2.wav,66149,0.346815,0.092243,0.132003,0.004620,1788.539719,111407.437613,2084.565132,...,52.42091,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues
3,1,blues.00000.3.wav,66149,0.363639,0.086856,0.132565,0.002448,1655.289045,111952.284517,1960.039988,...,52.42091,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues
4,1,blues.00000.4.wav,66149,0.335579,0.088129,0.143289,0.001701,1630.656199,79667.267654,1948.503884,...,52.42091,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9990 entries, 0 to 9989
Columns: 129 entries, id_cancion to label_30s
dtypes: float64(116), int64(7), object(6)
memory usage: 9.8+ MB


In [ ]:
# Comprobar que no se han perdido canciones ni trozos

print("Filas en features_3:", df_features_3_ml.shape[0])
print("Filas en df_macro:", df_macro.shape[0])

print("Canciones distintas en features_3:", df_features_3_ml['cancion_base'].nunique())
print("Canciones distintas en df_macro:", df_macro['id_cancion'].nunique())
ids_perdidos = set(df_modelo_30['id_cancion']) - set(df_macro['id_cancion'])

print("IDs perdidos al crear df_macro:", len(ids_perdidos))


Filas en features_3: 9990
Filas en df_macro: 9990
Canciones distintas en features_3: 1000
Canciones distintas en df_macro: 1000
IDs perdidos al crear df_macro: 0


In [ ]:
# Muestra las columnas principales para comprobar la unión

columnas_revision = [
    'id_cancion',
    'filename_3s',
    'filename_30s',
    'cancion_base',
    'cancion',
    'id_genero',
    'label_3s',
    'label_30s'
]

display(df_macro[columnas_revision].head(20))

,id_cancion,filename_3s,filename_30s,cancion_base,cancion,id_genero,label_3s,label_30s
0,1,blues.00000.0.wav,blues.00000.wav,blues.00000.wav,blues.00000.wav,1,blues,blues
1,1,blues.00000.1.wav,blues.00000.wav,blues.00000.wav,blues.00000.wav,1,blues,blues
2,1,blues.00000.2.wav,blues.00000.wav,blues.00000.wav,blues.00000.wav,1,blues,blues
3,1,blues.00000.3.wav,blues.00000.wav,blues.00000.wav,blues.00000.wav,1,blues,blues
4,1,blues.00000.4.wav,blues.00000.wav,blues.00000.wav,blues.00000.wav,1,blues,blues
5,1,blues.00000.5.wav,blues.00000.wav,blues.00000.wav,blues.00000.wav,1,blues,blues
6,1,blues.00000.6.wav,blues.00000.wav,blues.00000.wav,blues.00000.wav,1,blues,blues
7,1,blues.00000.7.wav,blues.00000.wav,blues.00000.wav,blues.00000.wav,1,blues,blues
8,1,blues.00000.8.wav,blues.00000.wav,blues.00000.wav,blues.00000.wav,1,blues,blues
9,1,blues.00000.9.wav,blues.00000.wav,blues.00000.wav,blues.00000.wav,1,blues,blues


In [ ]:
# Guardar la tabla en un archivo CSV

df_macro.to_csv(PREPROCESADOS / "df_macro.csv", index=False)

In [ ]:
# Revisar tamaño de la tabla

print("Filas:", df_macro.shape[0])
print("Columnas:", df_macro.shape[1])
ids_unicos = df_macro['id_cancion'].unique()

print("Total de IDs únicos:", len(ids_unicos))

Filas: 9990
Columnas: 129
Total de IDs únicos: 1000


In [ ]:
# Crear tabla de canción con features_3

df_modelo_3 = df_features_3_ml.merge(
    df_cancion,
    left_on='cancion_base',
    right_on='cancion',
    how='inner'
)
display(df_modelo_3.head())

,filename_3s,length_3s,chroma_stft_mean_3s,chroma_stft_var_3s,rms_mean_3s,rms_var_3s,spectral_centroid_mean_3s,spectral_centroid_var_3s,spectral_bandwidth_mean_3s,spectral_bandwidth_var_3s,...,label_3s,cancion_base,id_cancion,cancion,duracion,coste,id_genero,id_disco,id_autor,id_discografica
0,blues.00000.0.wav,66149,0.335406,0.091048,0.130405,0.003521,1773.065032,167541.630869,1972.744388,117335.771563,...,blues,blues.00000.wav,1,blues.00000.wav,138.0,744.01,1,1,1,1
1,blues.00000.1.wav,66149,0.343065,0.086147,0.112699,0.001450,1816.693777,90525.690866,2010.051501,65671.875673,...,blues,blues.00000.wav,1,blues.00000.wav,138.0,744.01,1,1,1,1
2,blues.00000.2.wav,66149,0.346815,0.092243,0.132003,0.004620,1788.539719,111407.437613,2084.565132,75124.921716,...,blues,blues.00000.wav,1,blues.00000.wav,138.0,744.01,1,1,1,1
3,blues.00000.3.wav,66149,0.363639,0.086856,0.132565,0.002448,1655.289045,111952.284517,1960.039988,82913.639269,...,blues,blues.00000.wav,1,blues.00000.wav,138.0,744.01,1,1,1,1
4,blues.00000.4.wav,66149,0.335579,0.088129,0.143289,0.001701,1630.656199,79667.267654,1948.503884,60204.020268,...,blues,blues.00000.wav,1,blues.00000.wav,138.0,744.01,1,1,1,1


In [ ]:
# Revisar resultado creado

print("Filas en features_3:", df_features_3_ml.shape[0])
print("Filas en df_modelo_3:", df_modelo_3.shape[0])

print("Canciones en df_cancion:", df_cancion['id_cancion'].nunique())
print("Canciones en df_modelo_3:", df_modelo_3['id_cancion'].nunique())

ids_perdidos = set(df_cancion['id_cancion']) - set(df_modelo_3['id_cancion'])

print("IDs perdidos:", len(ids_perdidos))

Filas en features_3: 9990
Filas en df_modelo_3: 9990
Canciones en df_cancion: 1000
Canciones en df_modelo_3: 1000
IDs perdidos: 0


In [ ]:
# Creamos DF de 'cancion' con features_3_sec.
df_modelo_3.to_csv(PREPROCESADOS / "df_cancion_3s.csv", index=False)

In [ ]:
# Crear tabla de features_3 con features_30

df_features_3_30 = df_features_3_ml.merge(
    df_features_30_ml,
    left_on='cancion_base',
    right_on='filename_30s',
    how='inner'
)

display(df_features_3_30.head())

,filename_3s,length_3s,chroma_stft_mean_3s,chroma_stft_var_3s,rms_mean_3s,rms_var_3s,spectral_centroid_mean_3s,spectral_centroid_var_3s,spectral_bandwidth_mean_3s,spectral_bandwidth_var_3s,...,mfcc16_var_30s,mfcc17_mean_30s,mfcc17_var_30s,mfcc18_mean_30s,mfcc18_var_30s,mfcc19_mean_30s,mfcc19_var_30s,mfcc20_mean_30s,mfcc20_var_30s,label_30s
0,blues.00000.0.wav,66149,0.335406,0.091048,0.130405,0.003521,1773.065032,167541.630869,1972.744388,117335.771563,...,52.42091,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues
1,blues.00000.1.wav,66149,0.343065,0.086147,0.112699,0.001450,1816.693777,90525.690866,2010.051501,65671.875673,...,52.42091,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues
2,blues.00000.2.wav,66149,0.346815,0.092243,0.132003,0.004620,1788.539719,111407.437613,2084.565132,75124.921716,...,52.42091,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues
3,blues.00000.3.wav,66149,0.363639,0.086856,0.132565,0.002448,1655.289045,111952.284517,1960.039988,82913.639269,...,52.42091,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues
4,blues.00000.4.wav,66149,0.335579,0.088129,0.143289,0.001701,1630.656199,79667.267654,1948.503884,60204.020268,...,52.42091,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues


In [ ]:
# Revisar resultado creado

print("Filas en features_3:", df_features_3_ml.shape[0])
print("Filas en df_features_3_30:", df_features_3_30.shape[0])

print("Canciones en features_3:", df_features_3_ml['cancion_base'].nunique())
print("Canciones en features_30:", df_features_30_ml['filename_30s'].nunique())
print("Canciones en df_features_3_30:", df_features_3_30['cancion_base'].nunique())

Filas en features_3: 9990
Filas en df_features_3_30: 9990
Canciones en features_3: 1000
Canciones en features_30: 1000
Canciones en df_features_3_30: 1000


In [ ]:
# La unión de fragmentos de 3 s con features de 30 s se guarda como df_30s_3.csv.
# La tabla de una fila por canción es df_modelo_30.csv, también guardada como df_cancion_30s.csv.
# Creamos DF de 'features_30' con 'features_3_sec'.
df_features_3_30.to_csv(PREPROCESADOS / "df_30s_3.csv", index=False)